<a href="https://colab.research.google.com/github/Suman-Poola/Random-Forrest-s-p500/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [38]:
from fastai import *
import pandas as pd
import os
import zipfile
import numpy as np
from dotenv import load_dotenv
import yfinance as yf
from sklearn.preprocessing import LabelEncoder
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kaggle


def build_dataset(time_period):


    dataset_id = 'darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily'
    path = 'ticker_list'
    kaggle.api.dataset_download_cli(dataset_id, path = str(path), unzip=True, force=True)

    tk = pd.read_csv('ticker_list/sp500_companies.csv')

    tickers = tk['symbol'].tolist()
    tickers[0:25]
    tickers = tickers[0:250]
    len(tickers)

    df = yf.download(ticker, period=time_period)

    closes = df["Close"]
    returns = closes.pct_change() *100

    volumes = df['Volume']

    cs = closes.stack().rename("Close")
    rs = returns.stack().rename("Returns")
    vs = volumes.stack().rename("Volume")
    fdf = pd.concat([cs,rs,vs], axis = 1).reset_index()

    fdf.columns = ['date', 'ticker', 'close', 'return', 'volume']
    fdf = fdf.dropna(subset=['return'])

    fdf.sort_values(by=['ticker', 'date'], inplace=True)
    fdf.sort_values(['ticker', 'date'], inplace=True)
    df=fdf
    len(df)

    df['Lag 1'] = df.groupby('ticker')['close'].shift(1)
    df['Lag 2'] = df.groupby('ticker')['close'].shift(2)
    df['Lag 3'] = df.groupby('ticker')['close'].shift(3)
    df['Lag 5'] = df.groupby('ticker')['close'].shift(5)
    df['SMA 5'] = df.groupby('ticker')['close'].transform(lambda s: s.rolling(window=5).mean())
    df['SMA 10'] = df.groupby('ticker')['close'].transform(lambda s: s.rolling(window=10).mean())
    df['SMA 20'] = df.groupby('ticker')['close'].transform(lambda s: s.rolling(window=20).mean())
    df['SMA 30'] = df.groupby('ticker')['close'].transform(lambda s: s.rolling(window=30).mean())
    df['SMA 50'] = df.groupby('ticker')['close'].transform(lambda s: s.rolling(window=50).mean())

    df['volatility 5'] = df.groupby('ticker')['return'].transform(lambda s: s.rolling(5).std())
    df['volatility 10'] = df.groupby('ticker')['return'].transform(lambda s: s.rolling(10).std())
    df['volatility 20'] = df.groupby('ticker')['return'].transform(lambda s: s.rolling(20).std())
    df['volatility 30'] = df.groupby('ticker')['return'].transform(lambda s: s.rolling(30).std())

    df['Price change per volume'] = 1000000*(df['return']/df['volume'])

    df['id'] = LabelEncoder().fit_transform(df['ticker'])

    df['mk cap'] = df['close'] * df['volume'] / 100000000
    df = df.dropna()

    len(df)
    df['future return'] = df.groupby('ticker')['return'].shift(-1)
    df = df.dropna()


    return df

In [39]:
df = build_dataset("36mo")

Dataset URL: https://www.kaggle.com/datasets/darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily
License(s): CC0-1.0


100%|██████████| 101M/101M [00:00<00:00, 196MB/s] 


/tmp/ipykernel_5888/362759800.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period=time_period)
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_5888/362759800.py:73: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['future return'] = df.groupby('ticker')['return'].shift(-1)


In [40]:
device = torch.device("cpu")

features = ['close', 'return', 'Lag 1', 'Lag 2', 'Lag 3', 'Lag 5', 'volatility 5', 'volatility 10', 'volatility 20', 'volatility 30','SMA 10' ,'SMA 20','SMA 30','SMA 50','volume','Price change per volume','id','mk cap']
x = df[features]
target = ['future return']

In [41]:
def make_sequences(window, ticker_col='ticker', date_col='date'):
    X_list, y_list = [], []

    for ticker, group in df.groupby(target):
        group = group.sort_values(date_col)
        feat_arr = group[features].values
        target_arr = group[ticker_col].values

        n_days = len(group)
        for i in range(n_days - window):
            X_list.append(feat_arr[i : i + window])
            y_list.append(target_arr[i + window])
    return np.array(X_list), np.array(y_list)

In [42]:
X_seq, y_seq = make_sequences(10)
X_seq.shape
y_seq.shape

(0,)

In [43]:
train_pct = 0.8
X_train, y_train = [], []
X_test, y_test = [], []
for ticker, group in df.groupby('ticker'):
    group = group.sort_values('date')
    feat_arr = group[features].values
    target_arr = group[target].values

    n_days = len(group)


    ticker_X, ticker_y = [], []
    for i in range(n_days - 10 ): #10 is the window, be sure to change it if you change the window
         ticker_X.append(feat_arr[i : i + 10])
         ticker_y.append(target_arr[i + 10])

    n_seqs = len(ticker_X)
    split_idx = int(train_pct * n_seqs)

    X_train.extend(ticker_X[:split_idx])
    y_train.extend(ticker_y[:split_idx])
    X_test.extend(ticker_X[split_idx:])
    y_test.extend(ticker_y[split_idx:])

x_train = np.array(X_train)
y_train = np.array(y_train)
x_test = np.array(X_test)
y_test = np.array(y_test)

print("train:", x_train.shape, y_train.shape)
print("test: ", x_test.shape, y_test.shape)

train: (553, 10, 18) (553, 1)
test:  (139, 10, 18) (139, 1)


In [44]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(553, 10, 18)
(553, 1)
(139, 10, 18)
(139, 1)


In [45]:
import numpy as np
for i, f in enumerate(features):
    n_inf = np.isinf(x_train[:, :, i]).sum()
    if n_inf > 0:
        print(f, n_inf)

In [46]:
id_idx = features.index('id')
cont_idx = [i for i, f in enumerate(features) if f != 'id']
n_cont = len(cont_idx)


scaler = StandardScaler()
scaler.fit(x_train[:, :, cont_idx].reshape(-1, n_cont))

x_train_scaled = scaler.transform(x_train[:, :, cont_idx].reshape(-1, n_cont)).reshape(x_train.shape[0], x_train.shape[1], n_cont)
x_test_scaled = scaler.transform(x_test[:, :, cont_idx].reshape(-1, n_cont)).reshape(x_test.shape[0], x_test.shape[1], n_cont)

id_train = x_train[:, 0, id_idx].astype(np.int64)
id_test = x_test[:, 0, id_idx].astype(np.int64)
n_tickers = int(max(id_train.max(), id_test.max())) + 1

print(x_train_scaled.shape, id_train.shape, n_tickers)

(553, 10, 17) (553,) 1


In [47]:
class StockSequenceDataset(Dataset):
    def __init__(self, X, ids, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.ids[idx], self.y[idx]

train_ds = StockSequenceDataset(x_train_scaled, id_train, y_train)
test_ds = StockSequenceDataset(x_test_scaled, id_test, y_test)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

xb, ib, yb = next(iter(train_loader))

In [48]:
class StockLSTM(nn.Module):
    def __init__(self, n_cont_features, n_tickers, emb_dim=16, hidden_size=64,
                 num_layers=1, dropout=0.3, emb_dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(n_tickers, emb_dim)
        self.emb_drop = nn.Dropout(emb_dropout)
        self.rnn = nn.LSTM(
            input_size=n_cont_features + emb_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,  # inter-layer dropout only applies when num_layers > 1
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x, ids):
        e = self.emb_drop(self.emb(ids))                      # (B, emb_dim)
        e = e.unsqueeze(1).expand(-1, x.size(1), -1)          # (B, T, emb_dim)
        out, _ = self.rnn(torch.cat([x, e], dim=-1))          # (B, T, hidden)
        last_step = self.drop(out[:, -1, :])
        return self.fc(last_step)

model = StockLSTM(n_cont_features=n_cont, n_tickers=n_tickers,
                  emb_dim=16, hidden_size=64, num_layers=1,
                  dropout=0.3, emb_dropout=0.1).to(device)

In [49]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 20
train_losses, test_losses = [], []

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for xb, ib, yb in train_loader:
        xb, ib, yb = xb.to(device), ib.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb, ib)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    train_loss = running_loss / len(train_ds)

    model.eval()
    running_test_loss = 0.0
    with torch.no_grad():
        for xb, ib, yb in test_loader:
            xb, ib, yb = xb.to(device), ib.to(device), yb.to(device)
            pred = model(xb, ib)
            loss = criterion(pred, yb)
            running_test_loss += loss.item() * xb.size(0)
    test_loss = running_test_loss / len(test_ds)

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    print(f"Epoch {epoch+1}  train {train_loss:.4f}  test {test_loss:.4f}")

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([256, 1, 1])) that is different to the input size (torch.Size([256, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([41, 1, 1])) that is different to the input size (torch.Size([41, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([139, 1, 1])) that is different to the input size (torch.Size([139, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  re

Epoch 1  train 2.8730  test 7.4858
Epoch 2  train 2.8286  test 7.3206
Epoch 3  train 2.8213  test 7.2653
Epoch 4  train 2.8290  test 7.2563
Epoch 5  train 2.8328  test 7.2666
Epoch 6  train 2.8293  test 7.2898
Epoch 7  train 2.8222  test 7.3134
Epoch 8  train 2.8209  test 7.3211
Epoch 9  train 2.8210  test 7.3189
Epoch 10  train 2.8211  test 7.3181
Epoch 11  train 2.8212  test 7.3208
Epoch 12  train 2.8240  test 7.3220
Epoch 13  train 2.8184  test 7.3150
Epoch 14  train 2.8196  test 7.3069
Epoch 15  train 2.8198  test 7.3072
Epoch 16  train 2.8198  test 7.3057
Epoch 17  train 2.8260  test 7.3079
Epoch 18  train 2.8243  test 7.3125
Epoch 19  train 2.8242  test 7.3128
Epoch 20  train 2.8240  test 7.3125
